# RoadVision AI - Dataset Quality Validation (Original Data)

Inspects the immutable source dataset (`trafic_data`). Checks file structure, pairing integrity, label formatting, class ID ranges, bounding box boundaries, and empty annotations without mutating source files.

In [1]:
import os
import math
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import yaml

# Configurable paths via environment variables or default user cache
DEFAULT_CACHE = Path.home() / ".cache" / "kagglehub" / "datasets" / "ashfakyeafi" / "road-vehicle-images-dataset" / "versions" / "2"
DATASET_VERSION_ROOT = Path(os.getenv("ROADVISION_DATASET_VERSION_ROOT", DEFAULT_CACHE))
ORIGINAL_ROOT = Path(os.getenv("ROADVISION_ORIGINAL_ROOT", DATASET_VERSION_ROOT / "trafic_data"))

REPORT_DIR = ORIGINAL_ROOT / "validation_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

print("Original Dataset Root:", ORIGINAL_ROOT.resolve())
print("Original Root Exists:", ORIGINAL_ROOT.exists())
if not ORIGINAL_ROOT.exists():
    raise FileNotFoundError(f"Dataset folder not found at: {ORIGINAL_ROOT}")

Original Dataset Root: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\trafic_data
Original Root Exists: True


### 1. Load Original YAML Class Configuration

In [2]:
yaml_path = ORIGINAL_ROOT / "data_1.yaml"
nc = None
class_names = []

if yaml_path.is_file():
    with yaml_path.open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file) or {}
    names_raw = config.get("names", [])
    if isinstance(names_raw, dict):
        class_names = [str(names_raw[k]) for k in sorted(names_raw, key=lambda x: int(x))]
    elif isinstance(names_raw, list):
        class_names = [str(n) for n in names_raw]
    nc = config.get("nc", len(class_names))
    print(f"Loaded YAML: {yaml_path.name}")
    print(f"Declared nc: {nc}")
    print(f"Class count parsed: {len(class_names)}")
    assert nc == len(class_names), f"nc ({nc}) does not match class_names length ({len(class_names)})"
else:
    raise FileNotFoundError(f"data_1.yaml not found at {yaml_path}")

Loaded YAML: data_1.yaml
Declared nc: 21
Class count parsed: 21


### 2. Check Directory Structure and Image-Label Pairing

In [3]:
split_rows = []
pairing_issues = []

for split in SPLITS:
    split_dir = ORIGINAL_ROOT / split
    image_dir = split_dir / "images"
    label_dir = split_dir / "labels"

    split_exists = split_dir.is_dir()
    img_exists = image_dir.is_dir()
    lbl_exists = label_dir.is_dir()

    if not split_exists:
        split_rows.append({
            "split": split, "split_folder_exists": False,
            "images_folder_exists": False, "labels_folder_exists": False,
            "images": 0, "label_files": 0,
            "images_without_label": 0, "labels_without_image": 0
        })
        pairing_issues.append({"split": split, "issue": "Missing split folder", "file": str(split_dir)})
        continue

    images = {p.stem.lower(): p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS} if img_exists else {}
    labels = {p.stem.lower(): p for p in label_dir.glob("*.txt") if p.is_file()} if lbl_exists else {}

    images_without_label = sorted(set(images.keys()) - set(labels.keys()))
    labels_without_image = sorted(set(labels.keys()) - set(images.keys()))

    for stem in images_without_label:
        pairing_issues.append({"split": split, "issue": "Image without label file", "file": str(images[stem])})
    for stem in labels_without_image:
        pairing_issues.append({"split": split, "issue": "Label without image", "file": str(labels[stem])})

    split_rows.append({
        "split": split,
        "split_folder_exists": True,
        "images_folder_exists": img_exists,
        "labels_folder_exists": lbl_exists,
        "images": len(images),
        "label_files": len(labels),
        "images_without_label": len(images_without_label),
        "labels_without_image": len(labels_without_image)
    })

split_summary_df = pd.DataFrame(split_rows)
pairing_issues_df = pd.DataFrame(pairing_issues, columns=["split", "issue", "file"])
display(split_summary_df)
print(f"Total pairing/structure issues detected: {len(pairing_issues_df)}")
display(pairing_issues_df.head(20))

,split,split_folder_exists,images_folder_exists,labels_folder_exists,images,label_files,images_without_label,labels_without_image
0,train,True,True,True,2704,2704,0,0
1,valid,True,True,True,300,300,0,0
2,test,False,False,False,0,0,0,0


Total pairing/structure issues detected: 1


,split,issue,file
0,test,Missing split folder,C:\Users\USER\.cache\kagglehub\datasets\ashfak...


### 3. Detailed YOLO Label and Bounding-Box Line Validation

In [4]:
annotation_issues = []
class_counts = Counter()
total_boxes = 0
empty_label_files = 0
labels_checked = 0

for split in SPLITS:
    label_dir = ORIGINAL_ROOT / split / "labels"
    if not label_dir.is_dir():
        continue

    for label_path in sorted(label_dir.glob("*.txt")):
        labels_checked += 1
        raw_text = label_path.read_text(encoding="utf-8", errors="replace")
        lines = raw_text.splitlines()
        nonempty_lines = [(idx, line.strip()) for idx, line in enumerate(lines, start=1) if line.strip()]

        if not nonempty_lines:
            empty_label_files += 1
            annotation_issues.append({
                "split": split, "file": str(label_path), "line": None,
                "issue": "Empty label file", "details": "File contains 0 annotation entries"
            })
            continue

        for line_no, line in nonempty_lines:
            parts = line.split()
            if len(parts) != 5:
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Malformed column count", "details": f"Found {len(parts)} tokens, expected 5"
                })
                continue

            try:
                cid_float = float(parts[0])
                if not math.isfinite(cid_float) or not cid_float.is_integer():
                    raise ValueError()
                class_id = int(cid_float)
            except (ValueError, OverflowError):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Invalid class ID format", "details": repr(parts[0])
                })
                continue

            try:
                coords = [float(x) for x in parts[1:]]
            except ValueError:
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Non-numeric coordinate", "details": " ".join(parts[1:])
                })
                continue

            if not all(math.isfinite(v) for v in coords):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Non-finite coordinate", "details": "Coordinates contain NaN or Inf"
                })
                continue

            if nc is not None and not (0 <= class_id < int(nc)):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Class ID out of range", "details": f"class_id={class_id}, range=[0, {nc-1}]"
                })

            x_center, y_center, width, height = coords
            if not (0.0 <= x_center <= 1.0 and 0.0 <= y_center <= 1.0):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Center outside bounds", "details": f"xc={x_center}, yc={y_center}"
                })

            if not (0.0 < width <= 1.0 and 0.0 < height <= 1.0):
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Invalid width/height", "details": f"w={width}, h={height}"
                })

            x_min, x_max = x_center - width / 2.0, x_center + width / 2.0
            y_min, y_max = y_center - height / 2.0, y_center + height / 2.0
            if x_min < 0.0 or x_max > 1.0 or y_min < 0.0 or y_max > 1.0:
                annotation_issues.append({
                    "split": split, "file": str(label_path), "line": line_no,
                    "issue": "Box outside image boundaries", "details": f"x=[{x_min:.4f}, {x_max:.4f}], y=[{y_min:.4f}, {y_max:.4f}]"
                })

            class_counts[class_id] += 1
            total_boxes += 1

issues_df = pd.DataFrame(annotation_issues, columns=["split", "file", "line", "issue", "details"])
print(f"Label files scanned: {labels_checked}")
print(f"Empty label files: {empty_label_files}")
print(f"Total bounding boxes parsed: {total_boxes}")
print(f"Total annotation anomalies: {len(issues_df)}")
display(issues_df.head(30))

Label files scanned: 3004
Empty label files: 2
Total bounding boxes parsed: 24348
Total annotation anomalies: 26


,split,file,line,issue,details
0,train,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,NaN,Empty label file,File contains 0 annotation entries
1,train,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,NaN,Empty label file,File contains 0 annotation entries
2,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,1.0,Invalid width/height,"w=0.0, h=0.0"
3,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,2.0,Invalid width/height,"w=0.0, h=0.0"
4,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,3.0,Invalid width/height,"w=0.0, h=0.0"
5,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,4.0,Invalid width/height,"w=0.0, h=0.0"
6,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,5.0,Invalid width/height,"w=0.0, h=0.0"
7,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,6.0,Invalid width/height,"w=0.0, h=0.0"
8,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,7.0,Invalid width/height,"w=0.0, h=0.0"
9,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,8.0,Invalid width/height,"w=0.0, h=0.0"


### 4. Original Dataset Class Distribution & Report Export

In [5]:
class_rows = [
    {"class_id": i, "class_name": class_names[i], "raw_bounding_boxes": class_counts.get(i, 0)}
    for i in range(len(class_names))
]
class_distribution_df = pd.DataFrame(class_rows)
display(class_distribution_df)

# Export reports
split_summary_df.to_csv(REPORT_DIR / "split_pairing_summary.csv", index=False)
pairing_issues_df.to_csv(REPORT_DIR / "pairing_issues.csv", index=False)
issues_df.to_csv(REPORT_DIR / "annotation_issues.csv", index=False)
class_distribution_df.to_csv(REPORT_DIR / "original_class_distribution.csv", index=False)

print("Validation reports saved to:", REPORT_DIR.resolve())

,class_id,class_name,raw_bounding_boxes
0,0,ambulance,70
1,1,army vehicle,43
2,2,auto rickshaw,372
3,3,bicycle,459
4,4,bus,3339
5,5,car,5474
6,6,garbagevan,3
7,7,human hauler,169
8,8,minibus,95
9,9,minivan,932


Validation reports saved to: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\trafic_data\validation_reports
